## Weighted Flow Map Geometry (D8-F1)

This notebook builds the line geometry for the weighted flow map, the primary network visualization for the report. For each of the 158 directed cross-border corridors in the Southeast Asian sexual-exploitation network, a straight line is constructed between the origin and destination country centroids, carrying the corridor's record volume as an attribute. The output is a GeoJSON file, symbolized separately in ArcGIS Pro. Line geometry is deliberately straight rather than curved (arc-implies-journey caveat, per the project design decisions), and the resulting map is geography and relations shown together, the direct counterpart to the attribute-only choropleths used elsewhere in the analysis.

In [2]:
import pandas as pd
import geopandas as gpd

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)

pandas: 2.3.3
geopandas: 1.1.4


### Step 1: Load inputs and confirm the join

The corridor edge list (`GEOG594A_D7_EdgeList.csv`, 158 directed origin-destination pairs with record counts) and the node centroid table (`GEOG594A_D8_F1_NodeCentroids.csv`, 71 countries with longitude/latitude) are loaded and checked against each other before any geometry is built, confirming every country referenced in the edge list has a matching centroid.

In [3]:
from pathlib import Path

EDGES = Path("outputs") / "GEOG594A_D7_EdgeList.csv"
CENTROIDS = Path("data") / "GEOG594A_D8_F1_NodeCentroids.csv"

edges_df = pd.read_csv(EDGES)
centroids_df = pd.read_csv(CENTROIDS)

print("edges:", edges_df.shape)
print(edges_df.columns.tolist())
print("centroids:", centroids_df.shape)
print(centroids_df.columns.tolist())

# confirm every country in the edge list has a centroid
edge_countries = set(edges_df["origin"]).union(edges_df["destination"])
centroid_countries = set(centroids_df["ISO3"])
missing = edge_countries - centroid_countries
print("countries in edges but missing a centroid:", missing)

edges: (158, 3)
['origin', 'destination', 'weight']
centroids: (71, 6)
['ISO3', 'ISO2', 'country_name', 'lon', 'lat', 'source']
countries in edges but missing a centroid: set()


### Step 2: Join centroid coordinates to each corridor

Each corridor's origin and destination country is matched to its centroid coordinates, producing an origin (lon, lat) and destination (lon, lat) pair for every one of the 158 corridors, the inputs a line-geometry needs.

In [4]:
# build lookup: ISO3 -> (lon, lat)
coord_lookup = centroids_df.set_index("ISO3")[["lon", "lat"]]

edges_geo = edges_df.merge(
    coord_lookup.rename(columns={"lon": "origin_lon", "lat": "origin_lat"}),
    left_on="origin", right_index=True, how="left"
).merge(
    coord_lookup.rename(columns={"lon": "dest_lon", "lat": "dest_lat"}),
    left_on="destination", right_index=True, how="left"
)

print(edges_geo.shape)
edges_geo.head()

(158, 7)


,origin,destination,weight,origin_lon,origin_lat,dest_lon,dest_lat
0,BDI,KHM,2,29.885182,-3.261252,105.039731,12.699187
1,BGD,KHM,8,90.432126,23.673729,105.039731,12.699187
2,BGR,KHM,3,25.251739,42.820437,105.039731,12.699187
3,BGR,PHL,1,25.251739,42.820437,121.822089,15.586542
4,BLR,MMR,5,27.964252,53.467914,97.088923,19.901228


### Step 3: Build line geometry and the thickness attribute

A `LineString` is constructed for each corridor, connecting its origin centroid to its destination centroid (straight lines, not arcs, per the routing-inference caveat already established for this project). A square-root-scaled thickness value is computed from each corridor's record count, the same transform locked for this figure's line width (checked against the real 158-corridor distribution before being chosen: heavily right-skewed, range 1-184).

In [5]:
import numpy as np
from shapely.geometry import LineString

edges_geo["geometry"] = edges_geo.apply(
    lambda r: LineString([(r["origin_lon"], r["origin_lat"]), (r["dest_lon"], r["dest_lat"])]),
    axis=1
)

edges_geo["thickness"] = np.sqrt(edges_geo["weight"])

print(edges_geo[["origin", "destination", "weight", "thickness"]].sort_values("weight", ascending=False).head(10))
print(edges_geo[["origin", "destination", "weight", "thickness"]].sort_values("weight").head(5))

    origin destination  weight  thickness
99     PHL         USA     184  13.564660
143    VNM         CHN     162  12.727922
149    VNM         MMR      97   9.848858
157    VNM         USA      79   8.888194
125    THA         USA      68   8.246211
150    VNM         MYS      61   7.810250
147    VNM         KHM      43   6.557439
30     IDN         MYS      41   6.403124
128    THA         ZAF      38   6.164414
91     PHL         JPN      31   5.567764
    origin destination  weight  thickness
35     IDN         UGA       1        1.0
113    THA         ARE       1        1.0
112    SRB         PHL       1        1.0
110    SOM         MMR       1        1.0
108    SLE         KHM       1        1.0


### Step 4: Build the GeoDataFrame and export GeoJSON

The corridor table is converted into a `GeoDataFrame` with a WGS84 (EPSG:4326) coordinate reference system, the standard lon/lat system ArcGIS Pro expects on import, then exported to GeoJSON (not shapefile, which truncates field names and has previously dropped the record-count field).

In [6]:
gdf = gpd.GeoDataFrame(
    edges_geo[["origin", "destination", "weight", "thickness", "geometry"]],
    geometry="geometry",
    crs="EPSG:4326"
)

print(gdf.shape)
print(gdf.crs)
gdf.head()

OUT = Path("outputs") / "GEOG594A_D8_F1_FlowLines.geojson"
gdf.to_file(OUT, driver="GeoJSON")
print("saved to", OUT)

(158, 5)
EPSG:4326
saved to outputs\GEOG594A_D8_F1_FlowLines.geojson


### Step 5: Verify the exported file

The exported GeoJSON is reloaded and checked against the source edge list: row count, CRS, and that no geometry or attribute was lost in the round trip.

In [7]:
check = gpd.read_file(OUT)

print("rows:", len(check))
print("CRS:", check.crs)
print("columns:", check.columns.tolist())
print("any missing geometry:", check.geometry.isna().sum())
print("weight sum matches original:", check["weight"].sum() == edges_df["weight"].sum())

rows: 158
CRS: EPSG:4326
columns: ['origin', 'destination', 'weight', 'thickness', 'geometry']
any missing geometry: 0
weight sum matches original: True


### Step 6: Re-express the geometry in Pacific-centered map coordinates

The corridor lines are transformed from geographic coordinates (EPSG:4326) into the WGS 1984 PDC Mercator projected system (EPSG:3832), the Pacific-centered projection used for the published map. Each line's endpoints are converted once, here, rather than being reprojected by the mapping software at draw time. This removes an artifact in which corridors whose endpoints fall on opposite sides of the projection's 30°W seam were rendered along the longer path and appeared detached from their origin country. Straight-line construction is unchanged; only the coordinate system in which the straightness is expressed differs.

In [8]:
gdf_pdc = gdf.to_crs("EPSG:3832")

print("CRS:", gdf_pdc.crs.name)
print("rows:", len(gdf_pdc))

# where each corridor's endpoints now sit, left to right, in map units
xs = gdf_pdc.geometry.apply(lambda g: g.coords[0][0])
xd = gdf_pdc.geometry.apply(lambda g: g.coords[1][0])
print("x range across all endpoints:", round(xs.min(), 0), "to", round(xd.max(), 0))

# any corridor still spanning more than half the world is worth eyeballing
half_world = 20037508.34
long_spans = (xs - xd).abs() > half_world
print("corridors spanning more than half the map:", long_spans.sum())
print(gdf_pdc.loc[long_spans, ["origin", "destination", "weight"]])

CRS: WGS 84 / PDC Mercator
rows: 158
x range across all endpoints: -18324399.0 to 16193381.0
corridors spanning more than half the map: 5
    origin destination  weight
5      BOL         KHM       2
46     KHM         ARG       3
65     LAO         ARG       3
117    THA         COL       7
142    VNM         ARG       5


### Step 7: Export the projected corridor geometry

The projected corridor lines are written to a GeoPackage, a format that preserves both the coordinate reference system and the full field names required downstream. The file is reloaded and checked against the source edge list on row count, coordinate system, and total corridor volume, confirming nothing was lost in the round trip. This file, rather than the geographic-coordinate version, is the input symbolized for the published map.

In [9]:
OUT_PDC = Path("outputs") / "GEOG594A_D8_F1_FlowLines_PDC.gpkg"

gdf_pdc.to_file(OUT_PDC, layer="flowlines", driver="GPKG")
print("saved to", OUT_PDC)

# reload and verify the round trip
check_pdc = gpd.read_file(OUT_PDC, layer="flowlines")

print("rows:", len(check_pdc))
print("CRS:", check_pdc.crs.name)
print("columns:", check_pdc.columns.tolist())
print("any missing geometry:", check_pdc.geometry.isna().sum())
print("weight sum matches original:", check_pdc["weight"].sum() == edges_df["weight"].sum())
print("thickness preserved:", round(check_pdc["thickness"].max(), 4))

saved to outputs\GEOG594A_D8_F1_FlowLines_PDC.gpkg
rows: 158
CRS: WGS 84 / PDC Mercator
columns: ['origin', 'destination', 'weight', 'thickness', 'geometry']
any missing geometry: 0
weight sum matches original: True
thickness preserved: 13.5647


### Step 8: Attach the corridor community attribute

Each corridor is labeled with the network community of the Southeast Asian country it touches, the attribute that drives line color on the published map. Where a corridor's origin is one of the seven Southeast Asian origin countries, that country's community is used; otherwise the destination's community is used. Origin takes precedence for corridors with both endpoints in the region. Community membership is carried over from the weighted Louvain partition, so line color is consistent with the node-link figure built from the same partition.

In [10]:
community_of = {
    "KHM": "Cambodia / Vietnam / Myanmar",
    "VNM": "Cambodia / Vietnam / Myanmar",
    "MMR": "Cambodia / Vietnam / Myanmar",
    "THA": "Thailand",
    "PHL": "Philippines",
    "IDN": "Indonesia / Laos",
    "LAO": "Indonesia / Laos",
}

def corridor_community(row):
    if row["origin"] in community_of:
        return community_of[row["origin"]]
    if row["destination"] in community_of:
        return community_of[row["destination"]]
    return "Other"

gdf_pdc["community"] = gdf_pdc.apply(corridor_community, axis=1)

sea = set(community_of)
both_ends = gdf_pdc.apply(lambda r: r["origin"] in sea and r["destination"] in sea, axis=1).sum()

print("corridors with no SEA endpoint (expect 0):", (gdf_pdc["community"] == "Other").sum())
print("corridors with both ends in SEA (expect 11):", both_ends)
print(gdf_pdc["community"].value_counts())

gdf_pdc.to_file(OUT_PDC, layer="flowlines", driver="GPKG")
check2 = gpd.read_file(OUT_PDC, layer="flowlines")
print("\nreloaded rows:", len(check2))
print("columns:", check2.columns.tolist())
print("weight sum still matches:", check2["weight"].sum() == edges_df["weight"].sum())

corridors with no SEA endpoint (expect 0): 0
corridors with both ends in SEA (expect 11): 11
community
Cambodia / Vietnam / Myanmar    65
Indonesia / Laos                33
Philippines                     31
Thailand                        29
Name: count, dtype: int64

reloaded rows: 158
columns: ['origin', 'destination', 'weight', 'thickness', 'community', 'geometry']
weight sum still matches: True
